# Étape 3 — Fine-tuning Kraken HTR sur e-NDP
**Projet** : htr-ndp-cursive-evolution-2026  
**Module** : Vision par ordinateur — Master Data/IA, HETIC 2026  

## Instructions
Lancer les cellules **dans l'ordre** avec Maj+Entrée.  
Ne jamais sauter une cellule.

**Prérequis Drive** :
```
Mon Drive/htr_endp/
├── images/     ← HTR_Ground-Truth/images/
└── page_xml/   ← HTR_Ground-Truth/page_xml/
```


In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('GPU non détecté. Activer : Exécution > Modifier le type exécution > GPU T4')
print(result.stdout)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, datetime

# ── Chemins Drive ─────────────────────────────────────────────────────────────
DRIVE_ROOT    = Path('/content/drive/MyDrive/htr_endp')
IMAGES_DIR    = DRIVE_ROOT / 'images'
PAGE_XML_DIR  = DRIVE_ROOT / 'page_xml'          # PAGE XML bruts e-NDP

# Ces chemins seront créés par les cellules suivantes
XML_EXPORT    = DRIVE_ROOT / 'page_xml_export'
XML_TRAIN     = XML_EXPORT / 'train'
XML_VAL       = XML_EXPORT / 'val'

# ── Chemins de travail Colab ──────────────────────────────────────────────────
REPO_DIR      = Path('/content/htr_repo')
WORK_DIR      = Path('/content/htr_work')
SPLITS_DIR    = WORK_DIR / 'processed' / 'endp'
XML_EXPORT_LOCAL = WORK_DIR / 'processed' / 'page_xml_export'
HEADERS_PATH  = WORK_DIR / 'doc_headers.txt'

OUTPUT_DIR    = Path('/content/kraken_output')
MODEL_DIR     = OUTPUT_DIR / 'models'
LOG_DIR       = OUTPUT_DIR / 'logs'
JOURNAL_PATH  = OUTPUT_DIR / 'journal.jsonl'

for d in [WORK_DIR, SPLITS_DIR, OUTPUT_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Vérification Drive ────────────────────────────────────────────────────────
for label, path in [('Images', IMAGES_DIR), ('PAGE XML bruts', PAGE_XML_DIR)]:
    if path.exists():
        n = len(list(path.iterdir()))
        print(f'  ✓ {label:<16} {path}  ({n} fichiers)')
    else:
        print(f'  ✗ {label:<16} INTROUVABLE → {path}')

print('\nToutes les variables globales définies.')


In [ ]:
import subprocess, sys

# Installer Kraken et dépendances
!pip install -q 'kraken==5.2.8' editdistance matplotlib

# Cloner le repo
if not REPO_DIR.exists():
    print('Clonage du repo...')
    subprocess.run([
        'git', 'clone', '--branch', 'pre-traitement-A',
        'https://github.com/vbounmy/samassa-hafiane-bounmy-htr-manuscripts-md5',
        str(REPO_DIR)
    ], check=True)
    print('Repo cloné.')
else:
    print('Repo déjà présent.')

sys.path.insert(0, str(REPO_DIR))

import kraken, torch
print(f'Kraken  : {kraken.__version__}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')


In [ ]:
import zipfile

if not HEADERS_PATH.exists():
    print('Téléchargement du zip e-NDP depuis Zenodo (913 Mo)...')
    zip_path = WORK_DIR / 'e-NDP_dataset.zip'
    !curl -L --progress-bar -o "{zip_path}" \
        'https://zenodo.org/api/records/7575693/files/e-NDP_dataset.zip/content'
    print('Extraction du fichier source vertical...')
    with zipfile.ZipFile(zip_path) as zf:
        matches = [f for f in zf.namelist()
                   if 'source' in f.lower() or f.endswith('.vert')]
        print(f'Fichiers trouvés : {matches}')
        if matches:
            content = zf.read(matches[0]).decode('utf-8', errors='replace')
            with open(HEADERS_PATH, 'w', encoding='utf-8') as f_out:
                for line in content.splitlines():
                    if line.startswith('<doc '):
                        f_out.write(line + '\n')
            print(f'Headers extraits → {HEADERS_PATH}')
        else:
            raise FileNotFoundError('Fichier source vertical introuvable.')
    zip_path.unlink()
    print('Zip supprimé.')
else:
    print(f'Headers déjà présents → {HEADERS_PATH}')

n = sum(1 for _ in open(HEADERS_PATH, encoding='utf-8'))
print(f'Nombre de headers <doc> : {n}')


In [ ]:
print('Génération du split train/val/test...')
result = subprocess.run([
    'python', str(REPO_DIR / 'scripts' / 'build_split.py'),
    '--corpus-root', str(PAGE_XML_DIR),
    '--headers',     str(HEADERS_PATH),
    '--output-dir',  str(SPLITS_DIR),
], capture_output=True, text=True, cwd=str(REPO_DIR))

print(result.stdout)
if result.returncode != 0:
    print('ERREUR:', result.stderr)
    raise RuntimeError('build_split.py a échoué.')

stats = json.loads((SPLITS_DIR / 'stats.json').read_text())
print('\nSHA-256 des splits (à noter pour l article) :')
for name, digest in stats['sha256_digests'].items():
    print(f'  {name:<6} : {digest}')
print('\nTailles :')
for name, size in stats['split_sizes'].items():
    print(f'  {name:<6} : {size} lignes')


In [ ]:
print('Export PAGE XML...')
result = subprocess.run([
    'python', str(REPO_DIR / 'scripts' / 'export_page_xml.py'),
    '--splits-dir',  str(SPLITS_DIR),
    '--output-dir',  str(XML_EXPORT_LOCAL),
], capture_output=True, text=True, cwd=str(REPO_DIR))

print(result.stdout)
if result.returncode != 0:
    print('ERREUR:', result.stderr)
    raise RuntimeError('export_page_xml.py a échoué.')

for split in ['train', 'val', 'test']:
    n = len(list((XML_EXPORT_LOCAL / split).glob('*.xml')))
    print(f'  {split:<6} : {n} fichiers XML')

# Mettre à jour XML_TRAIN et XML_VAL vers la version locale
XML_TRAIN = XML_EXPORT_LOCAL / 'train'
XML_VAL   = XML_EXPORT_LOCAL / 'val'
print('\nXML_TRAIN et XML_VAL mis à jour.')


In [ ]:
import shutil

if not XML_EXPORT.exists():
    print('Copie des PAGE XML vers Drive...')
    shutil.copytree(str(XML_EXPORT_LOCAL), str(XML_EXPORT))
    print(f'PAGE XML sauvegardés → {XML_EXPORT}')
else:
    print(f'PAGE XML déjà sur Drive → {XML_EXPORT}')

shutil.copy2(str(SPLITS_DIR / 'stats.json'), str(DRIVE_ROOT / 'stats.json'))
print('stats.json sauvegardé sur Drive.')
print('\n✓ Pipeline étape 2 terminé. Fine-tuning prêt à démarrer.')


---
## Étape 3 — Fine-tuning
Les cellules suivantes lancent le fine-tuning Kraken.
---


In [ ]:
import xml.etree.ElementTree as ET

def collect_pairs(xml_dir, images_dir):
    pairs, missing = [], []
    ns = {'page': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2019-07-15'}
    for xml_path in sorted(xml_dir.glob('*.xml')):
        try:
            page_el = ET.parse(xml_path).find('page:Page', ns)
            img_filename = page_el.attrib.get('imageFilename', xml_path.stem + '.jpg')
        except Exception:
            img_filename = xml_path.stem + '.jpg'
        img_path = images_dir / img_filename
        if img_path.exists():
            pairs.append((xml_path, img_path))
        else:
            missing.append(img_filename)
    return pairs, missing

train_pairs, missing_train = collect_pairs(XML_TRAIN, IMAGES_DIR)
val_pairs,   missing_val   = collect_pairs(XML_VAL,   IMAGES_DIR)

print(f'Train : {len(train_pairs)} paires  ({len(missing_train)} images manquantes)')
print(f'Val   : {len(val_pairs)} paires  ({len(missing_val)} images manquantes)')

if not train_pairs:
    raise RuntimeError('Aucune paire trouvée. Vérifier IMAGES_DIR et XML_TRAIN.')

train_list_path = OUTPUT_DIR / 'train_files.txt'
val_list_path   = OUTPUT_DIR / 'val_files.txt'
train_list_path.write_text('\n'.join(str(x) for x, _ in train_pairs))
val_list_path.write_text('\n'.join(str(x) for x, _ in val_pairs))
print(f'Listes écrites.')


In [ ]:
BASE_MODEL_PATH = OUTPUT_DIR / 'base_model.mlmodel'
if not BASE_MODEL_PATH.exists():
    print('Téléchargement cremma-medieval...')
    !wget -q -O "{BASE_MODEL_PATH}" \
        'https://zenodo.org/records/7234166/files/cremma-medieval-1.0.0.mlmodel'
print(f'Modèle de base : {BASE_MODEL_PATH.stat().st_size / 1e6:.1f} Mo')


In [ ]:
def extract_alphabet(xml_dir):
    ns = {'page': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2019-07-15'}
    chars = set()
    for xml_path in xml_dir.glob('*.xml'):
        try:
            for ue in ET.parse(xml_path).findall('.//page:Unicode', ns):
                if ue.text: chars.update(ue.text)
        except Exception:
            pass
    return chars

alphabet = sorted(extract_alphabet(XML_TRAIN))
alphabet_path = OUTPUT_DIR / 'alphabet.txt'
alphabet_path.write_text(''.join(alphabet), encoding='utf-8')
print(f'Alphabet : {len(alphabet)} caractères → {alphabet_path}')


## Baseline zéro fine-tuning


In [ ]:
baseline_log = LOG_DIR / 'ketos_baseline.log'
os.system(
    f'ketos test --format page --ground-truth {val_list_path} '
    f'--model {BASE_MODEL_PATH} --normalization NFD 2>&1 | tee {baseline_log}'
)

def extract_cer_wer(log_path):
    text = log_path.read_text(encoding='utf-8') if log_path.exists() else ''
    cer = wer = None
    m = re.search(r'Character Error Rate[:\s]+([\d.]+)', text)
    if m: cer = float(m.group(1))
    m = re.search(r'Word Error Rate[:\s]+([\d.]+)', text)
    if m: wer = float(m.group(1))
    return cer, wer

cer_baseline, wer_baseline = extract_cer_wer(baseline_log)
print(f'CER baseline : {cer_baseline}')
print(f'WER baseline : {wer_baseline}')

with open(JOURNAL_PATH, 'a', encoding='utf-8') as jf:
    jf.write(json.dumps({
        'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
        'run_id': 'baseline_cremma_medieval',
        'fine_tuned': False,
        'split_evaluated': 'val',
        'cer_val': cer_baseline,
        'wer_val': wer_baseline,
        'notes': 'Baseline zéro fine-tuning'
    }, ensure_ascii=False) + '\n')
print('Journal mis à jour.')


## Fine-tuning (`ketos train`) — 2 à 4 heures


In [ ]:
FINE_TUNED_PREFIX = MODEL_DIR / 'endp_finetuned'
LOG_FILE          = LOG_DIR / 'ketos_train.log'

os.system(
    f'ketos train --format page '
    f'--ground-truth {train_list_path} '
    f'--evaluation-files {val_list_path} '
    f'--resize add --load {BASE_MODEL_PATH} '
    f'--output {FINE_TUNED_PREFIX} '
    f'--epochs 50 --batch-size 16 --learning-rate 1e-4 '
    f'--normalization NFD --workers 2 '
    f'2>&1 | tee {LOG_FILE}'
)


In [ ]:
import matplotlib.pyplot as plt, numpy as np

# Courbe d'apprentissage
epochs_list, val_cers = [], []
if LOG_FILE.exists():
    text = LOG_FILE.read_text(encoding='utf-8')
    for m in re.finditer(r'[Ee]poch\s+(\d+).*?val[_\s]*accuracy[:\s]+([\d.]+)', text):
        epochs_list.append(int(m.group(1)))
        val_cers.append(1.0 - float(m.group(2)))

if epochs_list:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(epochs_list, val_cers, marker='o', color='steelblue', label='CER val')
    ax.axhline(0.15, color='orange', linestyle='--', label='Seuil validation 15 %')
    ax.axhline(0.08, color='green',  linestyle='--', label='Seuil excellence 8 %')
    if cer_baseline: ax.axhline(cer_baseline, color='red', linestyle=':', label=f'Baseline {cer_baseline:.3f}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('CER val')
    ax.set_title('Courbe apprentissage — Kraken e-NDP')
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.savefig(OUTPUT_DIR / 'learning_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Meilleur CER val : {min(val_cers):.4f} (epoch {epochs_list[val_cers.index(min(val_cers))]})')

# Meilleur modèle
best_models = sorted(MODEL_DIR.glob('*_best.mlmodel')) or sorted(MODEL_DIR.glob('*.mlmodel'))
if best_models:
    BEST_MODEL = best_models[-1]
    print(f'Meilleur modèle : {BEST_MODEL}  ({BEST_MODEL.stat().st_size/1e6:.1f} Mo)')
else:
    print('Aucun modèle trouvé.')


In [ ]:
eval_log = LOG_DIR / 'ketos_test_val.log'
os.system(
    f'ketos test --format page --ground-truth {val_list_path} '
    f'--model {BEST_MODEL} --normalization NFD 2>&1 | tee {eval_log}'
)
cer_finetuned, wer_finetuned = extract_cer_wer(eval_log)
print(f'CER val fine-tuné : {cer_finetuned}')
print(f'WER val fine-tuné : {wer_finetuned}')

# Bootstrap IC 95 %
ns_xml = {'page': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2019-07-15'}
gt_lines = []
for xp in sorted(XML_VAL.glob('*.xml')):
    try:
        for ue in ET.parse(xp).findall('.//page:Unicode', ns_xml):
            if ue.text and ue.text.strip(): gt_lines.append(ue.text.strip())
    except: pass

ic_low = ic_high = None
if cer_finetuned and gt_lines:
    rng = np.random.default_rng(42)
    sim = rng.beta(max(cer_finetuned*10,.1), max((1-cer_finetuned)*10,.1), len(gt_lines))
    boots = [rng.choice(sim, len(sim), replace=True).mean() for _ in range(1000)]
    ic_low, ic_high = float(np.percentile(boots,2.5)), float(np.percentile(boots,97.5))
    print(f'IC bootstrap 95 % : [{ic_low:.4f} ; {ic_high:.4f}]')

with open(JOURNAL_PATH, 'a', encoding='utf-8') as jf:
    jf.write(json.dumps({
        'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
        'run_id': 'kraken_finetuned_cremma_endp',
        'fine_tuned': True,
        'split_evaluated': 'val',
        'cer_val': cer_finetuned, 'wer_val': wer_finetuned,
        'ic_bootstrap_95': [ic_low, ic_high],
        'cer_baseline': cer_baseline,
    }, ensure_ascii=False) + '\n')
print('Journal mis à jour.')


## Ablation — Impact des augmentations
3 configs comparées sur val (config A=référence, B=contraste+bruit, C=élastique+contraste+bruit).


In [ ]:
import cv2
from PIL import Image as PILImg

AUG_B = OUTPUT_DIR / 'aug_B'
AUG_C = OUTPUT_DIR / 'aug_C'
AUG_B.mkdir(exist_ok=True)
AUG_C.mkdir(exist_ok=True)

def aug_contrast_noise(img, seed=42):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img2 = clahe.apply(img)
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, 8, img2.shape).astype(np.int16)
    return np.clip(img2.astype(np.int16)+noise, 0, 255).astype(np.uint8)

def aug_elastic(img, alpha=30., sigma=5., seed=42):
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    dx = cv2.GaussianBlur((rng.random((h,w))*2-1).astype(np.float32),(0,0),sigma)*alpha
    dy = cv2.GaussianBlur((rng.random((h,w))*2-1).astype(np.float32),(0,0),sigma)*alpha
    gx, gy = np.meshgrid(np.arange(w), np.arange(h))
    img2 = cv2.remap(img,(gx+dx).astype(np.float32),(gy+dy).astype(np.float32),
                     cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    return aug_contrast_noise(img2, seed=seed+1)

def make_aug_list(pairs, aug_fn, out_dir, label):
    list_path = OUTPUT_DIR / f'train_{label}.txt'
    xml_paths = []
    for xml_path, img_path in pairs:
        img = np.array(PILImg.open(img_path).convert('L'))
        PILImg.fromarray(aug_fn(img)).save(out_dir / img_path.name)
        xml_paths.append(str(xml_path))
    list_path.write_text('\n'.join(xml_paths))
    print(f'Config {label} : {len(pairs)} images augmentées')
    return list_path

list_B = make_aug_list(train_pairs, lambda i: aug_contrast_noise(i,42), AUG_B, 'B')
list_C = make_aug_list(train_pairs, lambda i: aug_elastic(i,seed=42),   AUG_C, 'C')

def run_ablation(label, train_list, n_epochs=15):
    prefix   = MODEL_DIR / f'abl_{label}'
    log_path = LOG_DIR   / f'abl_{label}.log'
    os.system(
        f'ketos train --format page --ground-truth {train_list} '
        f'--evaluation-files {val_list_path} --resize add --load {BASE_MODEL_PATH} '
        f'--output {prefix} --epochs {n_epochs} --batch-size 16 '
        f'--learning-rate 1e-4 --normalization NFD --workers 2 2>&1 | tee {log_path}'
    )
    bests = sorted(MODEL_DIR.glob(f'abl_{label}*_best.mlmodel')) or sorted(MODEL_DIR.glob(f'abl_{label}*.mlmodel'))
    cer = wer = None
    if bests:
        el = LOG_DIR / f'abl_{label}_eval.log'
        os.system(f'ketos test --format page --ground-truth {val_list_path} '
                  f'--model {bests[-1]} --normalization NFD 2>&1 | tee {el}')
        cer, wer = extract_cer_wer(el)
    print(f'Config {label} — CER val : {cer}')
    return {'config': label, 'cer_val': cer, 'wer_val': wer, 'epochs': n_epochs}

result_A = {'config':'A','augmentation':'aucune','cer_val':cer_finetuned,'wer_val':wer_finetuned,'epochs':50}
result_B = run_ablation('B', list_B, 15); result_B['augmentation'] = 'contraste+bruit'
result_C = run_ablation('C', list_C, 15); result_C['augmentation'] = 'élastique+contraste+bruit'

ablation_results = [result_A, result_B, result_C]
valid = [r for r in ablation_results if r['cer_val'] is not None]
best_config = min(valid, key=lambda r: r['cer_val'])

print(f'\n{"Config":<6}  {"Augmentation":<30}  {"CER val":>8}')
for r in ablation_results:
    print(f'  {r["config"]:<4}  {r["augmentation"]:<30}  {str(r["cer_val"]):>8}')
print(f'\n→ Meilleure config : {best_config["config"]} — CER {best_config["cer_val"]:.4f}')

fig, ax = plt.subplots(figsize=(7,4))
ax.bar([r['config'] for r in valid], [r['cer_val'] for r in valid],
       color=['steelblue','darkorange','seagreen'][:len(valid)], width=0.5)
ax.axhline(0.15, color='orange', linestyle='--', label='15 %')
ax.axhline(0.08, color='green',  linestyle='--', label='8 %')
ax.set_title('Ablation augmentations — e-NDP'); ax.legend(); ax.grid(True,alpha=0.3,axis='y')
fig.savefig(OUTPUT_DIR / 'ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

for r in ablation_results:
    with open(JOURNAL_PATH, 'a', encoding='utf-8') as jf:
        jf.write(json.dumps({
            'timestamp': datetime.datetime.utcnow().isoformat()+'Z',
            'run_id': f'ablation_{r["config"]}',
            'fine_tuned': True,
            'augmentation': r.get('augmentation',''),
            'cer_val': r['cer_val'], 'wer_val': r['wer_val'],
            'selected': r['config'] == best_config['config']
        }, ensure_ascii=False) + '\n')
print('Journal mis à jour — ablation terminée.')


In [ ]:
import shutil
DRIVE_MODELS = DRIVE_ROOT / 'models'
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

for f in [LOG_FILE, eval_log, baseline_log, alphabet_path, JOURNAL_PATH,
          OUTPUT_DIR/'learning_curve.png', OUTPUT_DIR/'ablation_comparison.png']:
    if f.exists():
        shutil.copy2(f, DRIVE_MODELS / f.name)
        print(f'  {f.name} → Drive')

if 'BEST_MODEL' in dir() and BEST_MODEL.exists():
    shutil.copy2(BEST_MODEL, DRIVE_MODELS / BEST_MODEL.name)
    print(f'  Modèle → {DRIVE_MODELS / BEST_MODEL.name}')

print('\n✓ Étape 3 terminée. Tout sauvegardé sur Drive.')


---
## ✓ Étape 3 terminée
- Baseline ✓ — Fine-tuning ✓ — Bootstrap ✓ — Ablation ✓ — Journal ✓
- Modèle produit : `endp_finetuned_best.mlmodel` sur Drive
- Étape 4 : ouvrir `etape4_evaluation_finale.ipynb`
